# Bra titel här

---

## 6 Oövervakad inlärning 

### Bakgrund (case)

Ledningen vill inte bara ha en modell - de vill också förstå om det finns **naturliga grupper av områden** i datan. Om sådana grupper finns kan det hjälpa till att:

- Segmentera områden (t.ex. "typområden" som liknar varandra)
- Upptäcka ovanliga områden (avvikare)
- Och få en enklare överblick i datan innan man tar beslut

### Uppdrag

Du ska därför undersöka om datan verkar innehålla struktur genom att använda **PCA eller KMEANS** på ett rimligt urval av X-variabler (inte target)

### Syfte

Att se om vi kan:

- Sammanfatta datan i ett enklare "mönster" (PCA) eller
- Hitta grupper av liknande områden (KMeans)
- Och diskutera hur detta skulle kunna användas som beslutsstöd

### Krav

- Implementera PCA **eller** KMeans
- Visa resultat (figur/tabell)
- Tolka kort: vad kan vi lära oss, och vad är osäkert?

Använd endast X-variabler (inte target) och motivera kort vilka features du inkluderade. Kom ihåg att metoderna är känsliga för skalning. 

### Importera bibliotek

In [2]:
# Grundläggande bibliotek för datahantering & visualisering

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Scikit-learn: config

from sklearn import set_config

# Scikit-learn: preprocessing & pipeline

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Scikit-learn: modeller

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Scikit-learn: clustering

from sklearn.cluster import KMeans

# Scikit-learn: dimension reduction

from sklearn.decomposition import PCA

# Scikit-learn: modellval & utvärdering

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    cross_val_predict,
    cross_val_score,
    train_test_split
)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    silhouette_score
)

In [ ]:
df_unsup = pd.read_csv("../data/housing.csv")

# df_unsup.head(5)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [ ]:
df_unsup = df_unsup.drop(columns=["median_house_value"])

# df_unsup.head(5)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,NEAR BAY


In [ ]:
df_unsup["rooms_per_household"] = df_unsup["total_rooms"] / df_unsup["households"]
df_unsup["bedrooms_per_room"] = df_unsup["total_bedrooms"] / df_unsup["total_rooms"]
df_unsup["population_per_household"] = df_unsup["population"] / df_unsup["households"]

df_unsup_clean = df_unsup.drop(columns=["total_rooms", "total_bedrooms", "population", "households"])

# df_unsup_clean.head(5)

,longitude,latitude,housing_median_age,median_income,median_house_value,ocean_proximity,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,8.3252,452600.0,NEAR BAY,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,8.3014,358500.0,NEAR BAY,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,7.2574,352100.0,NEAR BAY,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,5.6431,341300.0,NEAR BAY,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,3.8462,342200.0,NEAR BAY,6.281853,0.172096,2.181467


In [7]:
df_unsup_clean.describe()

,longitude,latitude,housing_median_age,median_income,median_house_value,rooms_per_household,bedrooms_per_room,population_per_household
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000
mean,-119.569704,35.631861,28.639486,3.870671,206855.816909,5.429000,0.213039,3.070655
std,2.003532,2.135952,12.585558,1.899822,115395.615874,2.474173,0.057983,10.386050
min,-124.350000,32.540000,1.000000,0.499900,14999.000000,0.846154,0.100000,0.692308
25%,-121.800000,33.930000,18.000000,2.563400,119600.000000,4.440716,0.175427,2.429741
50%,-118.490000,34.260000,29.000000,3.534800,179700.000000,5.229129,0.203162,2.818116
75%,-118.010000,37.710000,37.000000,4.743250,264725.000000,6.052381,0.239821,3.282261
max,-114.310000,41.950000,52.000000,15.000100,500001.000000,141.909091,1.000000,1243.333333
